<a href="https://colab.research.google.com/github/suchitapatil09/AI_LAB/blob/main/1BM24CS294_Week4_HillClimbing_N_Queens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

def calculate_cost(state):
    """Calculates the number of attacking pairs of queens (conflicts)."""
    cost = 0
    n = len(state)
    for i in range(n):
        for j in range(i + 1, n):
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                cost += 1
    return cost

def get_best_neighbor(state):
    """Finds the best neighbor with the minimum cost."""
    n = len(state)
    best_state = list(state)
    min_cost = calculate_cost(state)

    for col in range(n):
        original_row = state[col]
        for row in range(n):
            if row == original_row:
                continue
            state[col] = row
            cost = calculate_cost(state)

            if cost < min_cost:
                min_cost = cost
                best_state = list(state)

        state[col] = original_row

    return best_state, min_cost

def hill_climbing_with_restarts(n=4):
    """Repeatedly runs hill climbing with random restarts until cost 0 is found."""
    attempt = 1

    while True:
        # Generate a random starting board (e.g., [2, 0, 3, 1])
        current_state = [random.randint(0, n - 1) for _ in range(n)]
        current_cost = calculate_cost(current_state)

        print(f"Restart #{attempt}: Trying random start {current_state} (Cost: {current_cost})")

        while True:
            if current_cost == 0:
                print(f"\n SUCCESS! Found cost 0 solution: {current_state} on restart #{attempt}")
                return current_state

            next_state, next_cost = get_best_neighbor(current_state)

            # If we hit a local minimum and cost > 0, break and restart!
            if next_cost >= current_cost:
                print(f"   -> Stuck at local minimum {current_state} (Cost: {current_cost}). Restarting...")
                break

            current_state = next_state
            current_cost = next_cost

        attempt += 1

if __name__ == "__main__":
    hill_climbing_with_restarts(4)

Restart #1: Trying random start [0, 0, 0, 3] (Cost: 4)
   -> Stuck at local minimum [1, 2, 0, 3] (Cost: 1). Restarting...
Restart #2: Trying random start [1, 3, 0, 1] (Cost: 3)

 SUCCESS! Found cost 0 solution: [1, 3, 0, 2] on restart #2


In [ ]:
import random

def calculate_conflicts(board):
    """
    Heuristic function h(n): Calculates total pairs of attacking queens.
    0 means a valid solution.
    """
    conflicts = 0
    n = len(board)
    for i in range(n):
        for j in range(i + 1, n):
            # Same row conflict
            if board[i] == board[j]:
                conflicts += 1
            # Diagonal conflict: |row1 - row2| == |col1 - col2|
            elif abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1
    return conflicts

def evaluate_neighbors_matrix(board):
    """
    Prints a grid showing the cost (h-value) of placing a queen at
    each position (row, col), given the current state of other queens.
    Also shows detailed calculation for each neighbor.
    """
    n = len(board)
    print("\n" + "="*50)
    print(f"Current Board State: {board}  |  Current Cost h = {calculate_conflicts(board)}")
    print("="*50)

    # Show detailed calculation breakdown for neighbors
    print("\n--- Detailed Calculation of Neighboring Costs ---")
    for c in range(n):
        original_row = board[c]
        for r in range(n):
            if r == original_row:
                continue
            neighbor = list(board)
            neighbor[c] = r

            details = []
            for i in range(n):
                for j in range(i + 1, n):
                    if neighbor[i] == neighbor[j]:
                        details.append(f"Row conflict Q{i}(r{neighbor[i]})-Q{j}(r{neighbor[j]})")
                    elif abs(neighbor[i] - neighbor[j]) == abs(i - j):
                        details.append(f"Diag conflict Q{i}(r{neighbor[i]})-Q{j}(r{neighbor[j]})")

            cost = len(details)
            reason = ", ".join(details) if details else "No conflicts (h=0)"
            print(f"Move Col {c} -> Row {r} | State: {neighbor} | Cost: {cost} | Reason: {reason}")
    print("-" * 50)

    print("\nNeighboring Costs Grid (Row x Col):")
    print("      " + "  ".join([f"Col {c}" for c in range(n)]))
    print("   " + "-" * (7 * n))

    for r in range(n):
        row_str = f"Row {r} | "
        for c in range(n):
            if board[c] == r:
                cost = calculate_conflicts(board)
                row_str += f"[{cost}]   "
            else:
                neighbor = list(board)
                neighbor[c] = r
                cost = calculate_conflicts(neighbor)
                row_str += f" {cost}    "
        print(row_str)
    print("\nNote: Numbers in [ ] represent the CURRENT position of the queen.")

def get_best_neighbor_details(board):
    """
    Finds and prints the neighbor with the lowest heuristic cost.
    """
    n = len(board)
    current_cost = calculate_conflicts(board)
    best_board = list(board)
    best_cost = current_cost

    for col in range(n):
        original_row = board[col]
        for row in range(n):
            if row == original_row:
                continue

            neighbor = list(board)
            neighbor[col] = row
            cost = calculate_conflicts(neighbor)

            if cost < best_cost:
                best_cost = cost
                best_board = list(neighbor)

    return best_board, best_cost

def run_hill_climbing_with_restarts(n=4):
    """
    Runs Hill Climbing with Random Restarts until a global minimum (h = 0) is found.
    """
    restart_count = 0

    while True:
        restart_count += 1
        print(f"\n==========================================")
        print(f"       RESTART ATTEMPT #{restart_count}")
        print(f"==========================================")

        # Start with a fresh random board
        board = [random.randint(0, n - 1) for _ in range(n)]
        step = 0

        while True:
            step += 1
            print(f"\n>>> STEP {step} (Attempt #{restart_count}) <<<")

            evaluate_neighbors_matrix(board)

            current_cost = calculate_conflicts(board)
            best_neighbor, best_cost = get_best_neighbor_details(board)

            if best_cost >= current_cost:
                if current_cost == 0:
                    print(f"\n SUCCESS: Reached global minimum (h = 0) on Restart #{restart_count}!")
                    print(f"Final Solution Board: {board}")
                    return board
                else:
                    print(f"\n STUCK IN LOCAL MINIMUM (Current h = {current_cost}, Best neighbor h = {best_cost}).")
                    print("Restarting with a new random board...")
                    break

            print(f"\nMoving from state {board} (h={current_cost}) -> state {best_neighbor} (h={best_cost})")
            board = best_neighbor

if __name__ == "__main__":
    run_hill_climbing_with_restarts(n=4)


       RESTART ATTEMPT #1

>>> STEP 1 (Attempt #1) <<<

Current Board State: [0, 3, 3, 2]  |  Current Cost h = 2

--- Detailed Calculation of Neighboring Costs ---
Move Col 0 -> Row 1 | State: [1, 3, 3, 2] | Cost: 3 | Reason: Diag conflict Q0(r1)-Q2(r3), Row conflict Q1(r3)-Q2(r3), Diag conflict Q2(r3)-Q3(r2)
Move Col 0 -> Row 2 | State: [2, 3, 3, 2] | Cost: 4 | Reason: Diag conflict Q0(r2)-Q1(r3), Row conflict Q0(r2)-Q3(r2), Row conflict Q1(r3)-Q2(r3), Diag conflict Q2(r3)-Q3(r2)
Move Col 0 -> Row 3 | State: [3, 3, 3, 2] | Cost: 4 | Reason: Row conflict Q0(r3)-Q1(r3), Row conflict Q0(r3)-Q2(r3), Row conflict Q1(r3)-Q2(r3), Diag conflict Q2(r3)-Q3(r2)
Move Col 1 -> Row 0 | State: [0, 0, 3, 2] | Cost: 3 | Reason: Row conflict Q0(r0)-Q1(r0), Diag conflict Q1(r0)-Q3(r2), Diag conflict Q2(r3)-Q3(r2)
Move Col 1 -> Row 1 | State: [0, 1, 3, 2] | Cost: 2 | Reason: Diag conflict Q0(r0)-Q1(r1), Diag conflict Q2(r3)-Q3(r2)
Move Col 1 -> Row 2 | State: [0, 2, 3, 2] | Cost: 3 | Reason: Diag confli